# Notebook 3 - Interpretation des resultats et optimisation finale

Ce notebook repond a trois objectifs :

1. expliquer ce que representent les metriques (RMSE, MAE, R2) ;
2. comparer les familles de modeles de maniere lisible ;
3. optimiser les hyperparametres du modele final (avec focus Random Forest).

In [1]:
import os
import sys
from pathlib import Path

# Ajoute la racine du projet pour permettre les imports du package agritech.
_root = Path(os.getcwd())
while not (_root / "agritech").is_dir() and _root != _root.parent:
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
_root

WindowsPath('G:/GITHUB/Data-Scientist-OC/PROJET12')

In [2]:
import json

import mlflow
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_validate, train_test_split

from agritech.config import MLRUNS_DIR
from agritech.models.train import FEATURE_COLUMNS, TARGET_COLUMN, build_pipeline, load_training_dataset

## Comment lire les metriques

- RMSE : erreur quadratique moyenne racine. Plus faible = meilleur. Penalise fortement les grosses erreurs.
- MAE : erreur absolue moyenne. Plus faible = meilleur. Plus simple a interpreter en unite du rendement.
- R2 : proportion de variance expliquee. Plus proche de 1 = meilleur.

Interpretation rapide :
- RMSE > MAE est normal (la RMSE amplifie les grosses erreurs).
- Si R2 augmente et RMSE/MAE baissent, le modele progresse vraiment.
- On choisit en priorite le meilleur score de validation croisee pour limiter le sur-apprentissage.

In [3]:
dataset = load_training_dataset()
features = dataset[FEATURE_COLUMNS]
target = dataset[TARGET_COLUMN]

x_train, x_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.2,
    random_state=42,
)

dataset.shape, x_train.shape, x_test.shape

((38382, 7), (30705, 6), (7677, 6))

In [4]:
def holdout_metrics(pipeline):
    preds = pipeline.predict(x_test)
    return {
        'rmse': float(np.sqrt(mean_squared_error(y_test, preds))),
        'mae': float(mean_absolute_error(y_test, preds)),
        'r2': float(r2_score(y_test, preds)),
    }

def cv_rmse(pipeline):
    scores = cross_validate(
        pipeline,
        x_train,
        y_train,
        cv=5,
        scoring={'rmse': 'neg_root_mean_squared_error', 'mae': 'neg_mean_absolute_error', 'r2': 'r2'},
        n_jobs=-1,
        return_train_score=False,
    )
    return {
        'cv_rmse': float(-scores['test_rmse'].mean()),
        'cv_mae': float(-scores['test_mae'].mean()),
        'cv_r2': float(scores['test_r2'].mean()),
    }

base_models = {
    'linear_regression': LinearRegression(),
    'gradient_boosting': GradientBoostingRegressor(random_state=42),
    'random_forest': RandomForestRegressor(random_state=42, n_jobs=-1),
}

rows = []
for model_name, estimator in base_models.items():
    pipe = build_pipeline(estimator)
    cv_scores = cv_rmse(pipe)
    pipe.fit(x_train, y_train)
    test_scores = holdout_metrics(pipe)
    rows.append({'model': model_name, **cv_scores, **test_scores})

baseline_comparison = pd.DataFrame(rows).sort_values('cv_rmse').reset_index(drop=True)
baseline_comparison

,model,cv_rmse,cv_mae,cv_r2,rmse,mae,r2
0,random_forest,8695.443689,2988.155663,0.992055,7746.376497,2653.085770,0.993677
1,gradient_boosting,30037.640835,19442.602540,0.905376,30730.310229,20051.789102,0.900491
2,linear_regression,47318.985143,33984.267768,0.765408,46985.008353,33734.023092,0.767380


## Pourquoi Random Forest est souvent un bon candidat ici

Sur ce type de donnees (variables meteo + pesticides + effets pays/culture encodes), Random Forest est souvent pertinent car :

- il capture des relations non lineaires ;
- il gere bien les interactions entre variables ;
- il est robuste au bruit et a une normalisation imparfaite ;
- il est stable avec un bon compromis biais/variance apres tuning.

La decision finale reste data-driven : on garde le modele avec le meilleur compromis CV + test.

In [5]:
rf_stage1 = RandomizedSearchCV(
    estimator=build_pipeline(RandomForestRegressor(random_state=42, n_jobs=-1)),
    param_distributions={
        'model__n_estimators': [150, 250, 350, 500, 700],
        'model__max_depth': [10, 15, 20, 30, None],
        'model__min_samples_split': [2, 4, 6, 10],
        'model__min_samples_leaf': [1, 2, 4],
        'model__max_features': ['sqrt', 'log2', 0.6, 0.8],
        'model__bootstrap': [True, False],
    },
    n_iter=20,
    scoring='neg_root_mean_squared_error',
    cv=5,
    random_state=42,
    n_jobs=-1,
    refit=True,
    verbose=2,
)

rf_stage1.fit(x_train, y_train)
rf_stage1_best_params = rf_stage1.best_params_
rf_stage1_best_rmse = -rf_stage1.best_score_
rf_stage1_best_params, rf_stage1_best_rmse

Fitting 5 folds for each of 20 candidates, totalling 100 fits


({'model__n_estimators': 700,
  'model__min_samples_split': 4,
  'model__min_samples_leaf': 1,
  'model__max_features': 'sqrt',
  'model__max_depth': None,
  'model__bootstrap': False},
 np.float64(8672.289622101516))

In [6]:
def around_int(v, low_step, high_step, minimum=1):
    return sorted({max(minimum, v - low_step), max(minimum, v), max(minimum, v + high_step)})

n_estimators_best = int(rf_stage1_best_params['model__n_estimators'])
max_depth_best = rf_stage1_best_params['model__max_depth']
min_split_best = int(rf_stage1_best_params['model__min_samples_split'])
min_leaf_best = int(rf_stage1_best_params['model__min_samples_leaf'])
max_features_best = rf_stage1_best_params['model__max_features']
bootstrap_best = rf_stage1_best_params['model__bootstrap']

rf_refined_grid = {
    'model__n_estimators': around_int(n_estimators_best, 100, 100, minimum=50),
    'model__max_depth': [max_depth_best] if max_depth_best is None else around_int(int(max_depth_best), 5, 5, minimum=2),
    'model__min_samples_split': around_int(min_split_best, 1, 2, minimum=2),
    'model__min_samples_leaf': around_int(min_leaf_best, 1, 1, minimum=1),
    'model__max_features': [max_features_best],
    'model__bootstrap': [bootstrap_best],
}

rf_stage2 = GridSearchCV(
    estimator=build_pipeline(RandomForestRegressor(random_state=42, n_jobs=-1)),
    param_grid=rf_refined_grid,
    scoring='neg_root_mean_squared_error',
    cv=5,
    n_jobs=-1,
    refit=True,
    verbose=1,
)

rf_stage2.fit(x_train, y_train)
rf_stage2_best_params = rf_stage2.best_params_
rf_stage2_best_rmse = -rf_stage2.best_score_
rf_stage2_best_params, rf_stage2_best_rmse

Fitting 5 folds for each of 18 candidates, totalling 90 fits


({'model__bootstrap': False,
  'model__max_depth': None,
  'model__max_features': 'sqrt',
  'model__min_samples_leaf': 1,
  'model__min_samples_split': 3,
  'model__n_estimators': 800},
 np.float64(8590.90969131883))

In [7]:
rf_final_model = rf_stage2.best_estimator_
rf_final_test = holdout_metrics(rf_final_model)

best_baseline = baseline_comparison.iloc[0]
comparison_summary = pd.DataFrame([
    {
        'scenario': f"baseline_best::{best_baseline['model']}",
        'cv_rmse': float(best_baseline['cv_rmse']),
        'test_rmse': float(best_baseline['rmse']),
        'test_mae': float(best_baseline['mae']),
        'test_r2': float(best_baseline['r2']),
    },
    {
        'scenario': 'rf_stage1_random_search',
        'cv_rmse': float(rf_stage1_best_rmse),
        'test_rmse': float(holdout_metrics(rf_stage1.best_estimator_)['rmse']),
        'test_mae': float(holdout_metrics(rf_stage1.best_estimator_)['mae']),
        'test_r2': float(holdout_metrics(rf_stage1.best_estimator_)['r2']),
    },
    {
        'scenario': 'rf_stage2_grid_refinement',
        'cv_rmse': float(rf_stage2_best_rmse),
        'test_rmse': float(rf_final_test['rmse']),
        'test_mae': float(rf_final_test['mae']),
        'test_r2': float(rf_final_test['r2']),
    },
]).sort_values('cv_rmse').reset_index(drop=True)

comparison_summary

,scenario,cv_rmse,test_rmse,test_mae,test_r2
0,rf_stage2_grid_refinement,8590.909691,7560.176435,2978.997453,0.993977
1,rf_stage1_random_search,8672.289622,7713.393519,3094.570907,0.993731
2,baseline_best::random_forest,8695.443689,7746.376497,2653.085770,0.993677


In [8]:
MLRUNS_DIR.mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri(MLRUNS_DIR.as_uri())
mlflow.set_experiment('agritech-yield-final-interpretation')

with mlflow.start_run(run_name='rf_two_stage_optimization'):
    mlflow.log_param('model_name', 'random_forest')
    mlflow.log_param('optimization_strategy', 'RandomizedSearchCV_then_GridSearchCV')
    mlflow.log_param('feature_columns', ','.join(FEATURE_COLUMNS))
    mlflow.log_param('stage1_n_iter', 20)
    mlflow.log_param('cv', 5)
    mlflow.log_params({f'stage1_{k}': v for k, v in rf_stage1_best_params.items()})
    mlflow.log_params({f'stage2_{k}': v for k, v in rf_stage2_best_params.items()})
    mlflow.log_metric('stage1_cv_rmse', float(rf_stage1_best_rmse))
    mlflow.log_metric('stage2_cv_rmse', float(rf_stage2_best_rmse))
    mlflow.log_metric('final_test_rmse', float(rf_final_test['rmse']))
    mlflow.log_metric('final_test_mae', float(rf_final_test['mae']))
    mlflow.log_metric('final_test_r2', float(rf_final_test['r2']))

final_report = {
    'best_baseline_model': str(best_baseline['model']),
    'rf_stage1_best_params': rf_stage1_best_params,
    'rf_stage2_best_params': rf_stage2_best_params,
    'rf_stage2_test_metrics': rf_final_test,
}

print(json.dumps(final_report, indent=2, default=str))

2026/06/12 12:48:37 INFO mlflow.tracking.fluent: Experiment with name 'agritech-yield-final-interpretation' does not exist. Creating a new experiment.


{
  "best_baseline_model": "random_forest",
  "rf_stage1_best_params": {
    "model__n_estimators": 700,
    "model__min_samples_split": 4,
    "model__min_samples_leaf": 1,
    "model__max_features": "sqrt",
    "model__max_depth": null,
    "model__bootstrap": false
  },
  "rf_stage2_best_params": {
    "model__bootstrap": false,
    "model__max_depth": null,
    "model__max_features": "sqrt",
    "model__min_samples_leaf": 1,
    "model__min_samples_split": 3,
    "model__n_estimators": 800
  },
  "rf_stage2_test_metrics": {
    "rmse": 7560.176434709961,
    "mae": 2978.9974526996225,
    "r2": 0.9939772766482966
  }
}


## Conclusion a tirer

1. On compare d'abord plusieurs familles de modeles de facon neutre (Linear Regression, Gradient Boosting, Random Forest).
2. On regarde prioritairement le CV RMSE pour choisir un candidat robuste.
3. Random Forest est retenu si son CV RMSE est le plus bas (ou tres proche du meilleur avec meilleure stabilite test).
4. L'optimisation en deux etapes (Randomized puis Grid) affine les hyperparametres et consolide la performance finale.

En resume : si Random Forest sort gagnant dans `comparison_summary`, le choix est justifie par ses performances et sa robustesse sur des relations non lineaires entre variables agro-climatiques.